In [33]:
import json
import ast
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

def visualize_result(result, idx, save_dir="outputs"):
    os.makedirs(save_dir, exist_ok=True)

    # Load input image
    img_path = result["image_path"]
    img = mpimg.imread(img_path)

    ground_truth = ""#result["ground_truth"]
    prediction = result["model_output"]

    # Decide how many ranks to show (top 3)
    n_ranks = min(3, len(result["top_concepts_over_sequence"]))
    # Max number of crops across those ranks
    max_crops = max(len(ast.literal_eval(c["image_grounding_path"])) 
                    if isinstance(c["image_grounding_path"], str) else len(c["image_grounding_path"]) 
                    for c in result["top_concepts_over_sequence"][:n_ranks])

    # Layout: [big input image] + [similarity bar + crops...]
    fig = plt.figure(figsize=(4 + (max_crops+1)*2, n_ranks*2.8))
    gs = fig.add_gridspec(
        n_ranks, max_crops+2, 
        width_ratios=[2] + [0.3] + [1]*max_crops,
        wspace=0.05, hspace=0.05  # reduce spacing between cells
    )

    # Left side: input image (spans all rows)
    ax_left = fig.add_subplot(gs[:, 0])
    ax_left.imshow(img)
    ax_left.axis("off")
    match = "✅" if prediction.strip().lower() == ground_truth.strip().lower() else "❌"
    ax_left.set_title(f"GT: {ground_truth} | Pred: {prediction} {match}", fontsize=24)

    # Right side: rows for top ranks
    for rank in range(n_ranks):
        concept = result["top_concepts_over_sequence"][rank]
        paths_str = concept["image_grounding_path"]
        paths = ast.literal_eval(paths_str) if isinstance(paths_str, str) else paths_str
        similarity = concept.get("similarity", 0.0)

        # Add vertical similarity bar at column 1 (thin + orange)
        ax_bar = fig.add_subplot(gs[rank, 1])
        ax_bar.bar([0], [similarity], color="orange", width=0.2)
        ax_bar.set_ylim(0, 1)
        ax_bar.set_xticks([])
        ax_bar.set_yticks([0, 0.5, 1.0])
        ax_bar.tick_params(axis="y", labelsize=14)
        ax_bar.set_ylabel(f"{similarity:.2f}", fontsize=18, rotation=0, labelpad=15)
        ax_bar.set_title(f"Rank {rank+1}", fontsize=20, pad=8)

        # Add concept crops to the right
        for j, item in enumerate(paths):
            try:
                _, crop_path = item.split("@")
            except ValueError:
                crop_path = item

            ax = fig.add_subplot(gs[rank, j+2])
            if os.path.exists(crop_path):
                crop_img = mpimg.imread(crop_path)
                ax.imshow(crop_img)
            ax.axis("off")

        # Row label (text_grounding)
        text_label = ", ".join(concept.get("text_grounding", []))
        fig.text(0.35, 1 - (rank+1)/(n_ranks+0.2), text_label,
                 ha="left", va="center", fontsize=22, color="darkblue")

    # Make figure compact
    plt.subplots_adjust(wspace=0.05, hspace=0.05)
    save_path = os.path.join(save_dir, f"viz_{idx}.png")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved visualization: {save_path}")





def visualize_all(json_path, save_dir="outputs"):
    """
    Loop through all results in the JSON and visualize them.
    """
    with open(json_path, "r") as f:
        data = json.load(f)

    results = data["results"]
    for idx, result in enumerate(results):
        visualize_result(result, idx, save_dir)


# Example usage:
# visualize_all("your_file.json", save_dir="viz_outputs")


In [34]:
visualize_all("/mnt/abka03/Projects/xl-vlms/outputs/imagenet_5_class_qwen_cgdl_imagenet_unsupervised/explanations/snmf/vlm_explanations.json", save_dir="/mnt/abka03/Projects/xl-vlms/outputs/imagenet_5_class_qwen_cgdl_imagenet_unsupervised/plots")

/tmp/ipykernel_1225742/2682791391.py:77: UserWarning: Glyph 10060 (\N{CROSS MARK}) missing from font(s) DejaVu Sans.
  plt.savefig(save_path, dpi=150, bbox_inches="tight")


Saved visualization: /mnt/abka03/Projects/xl-vlms/outputs/imagenet_5_class_qwen_cgdl_imagenet_unsupervised/plots/viz_0.png
Saved visualization: /mnt/abka03/Projects/xl-vlms/outputs/imagenet_5_class_qwen_cgdl_imagenet_unsupervised/plots/viz_1.png
Saved visualization: /mnt/abka03/Projects/xl-vlms/outputs/imagenet_5_class_qwen_cgdl_imagenet_unsupervised/plots/viz_2.png
Saved visualization: /mnt/abka03/Projects/xl-vlms/outputs/imagenet_5_class_qwen_cgdl_imagenet_unsupervised/plots/viz_3.png
Saved visualization: /mnt/abka03/Projects/xl-vlms/outputs/imagenet_5_class_qwen_cgdl_imagenet_unsupervised/plots/viz_4.png
Saved visualization: /mnt/abka03/Projects/xl-vlms/outputs/imagenet_5_class_qwen_cgdl_imagenet_unsupervised/plots/viz_5.png
Saved visualization: /mnt/abka03/Projects/xl-vlms/outputs/imagenet_5_class_qwen_cgdl_imagenet_unsupervised/plots/viz_6.png
Saved visualization: /mnt/abka03/Projects/xl-vlms/outputs/imagenet_5_class_qwen_cgdl_imagenet_unsupervised/plots/viz_7.png
Saved visualizat

In [7]:
import torch

# Replace with your checkpoint path
path = "/mnt/abka03/Projects/xl-vlms/outputs/imagenet_5_class_qwen_dl_imagenet_unsupervised_description/concept/snmf/combined_concept_snmf_raw.pth"

# Load checkpoint
checkpoint = torch.load(path, map_location="cpu")

# Show top-level keys
print("Keys in checkpoint:", checkpoint.keys())
print(len(checkpoint["image_grounding_paths"]))

Keys in checkpoint: dict_keys(['concepts', 'activations', 'decomposition_method', 'text_grounding', 'image_grounding_paths', 'analysis_model'])
1000


In [2]:
import os
import random
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

def save_grounding_rows(image_grounding_paths, text_groundings, save_dir="groundings", max_rows=None):
    os.makedirs(save_dir, exist_ok=True)

    num_rows = len(image_grounding_paths)
    if max_rows:
        num_rows = min(num_rows, max_rows)

    for i in range(num_rows):
        tokens = text_groundings[i]
        paths = image_grounding_paths[i]

        # split path into prefix and suffix
        prefixes = [p.split("@")[0] for p in paths]
        suffixes = [p.split("@")[-1] for p in paths]

        # make a safe filename from unique prefixes
        unique_prefixes = sorted(set(prefixes))
        name_str = "_".join(unique_prefixes)
        name_str = name_str.replace("/", "_").replace(" ", "_")  # sanitize filename
        if len(name_str) > 100:  # avoid overly long filenames
            name_str = name_str[:100]

        # add random number to avoid overwriting
        rand_suffix = random.randint(1000, 9999)
        filename = f"{name_str}_{rand_suffix}.pdf"

        # figure size proportional to number of images (compact)
        fig, axes = plt.subplots(
            1, len(suffixes) + 1,
            figsize=(3*(len(suffixes)+1), 3),
            constrained_layout=True
        )

        # if only one row of axes, make iterable
        if len(suffixes) + 1 == 1:
            axes = [axes]

        # column 0: tokens as text
        axes[0].axis("off")
        axes[0].text(0.0, 0.5, "\n".join(tokens), fontsize=40,
                     va="center", ha="left", wrap=True)

        # columns 1..: images
        for j, ax in enumerate(axes[1:]):
            ax.axis("off")
            if os.path.exists(suffixes[j]):
                img = mpimg.imread(suffixes[j])
                ax.imshow(img)
            else:
                ax.text(0.5, 0.5, f"Missing:\n{suffixes[j]}",
                        ha="center", va="center", fontsize=10)

        # save without extra whitespace
        out_path = os.path.join(save_dir, filename)
        plt.savefig(out_path, dpi=150, bbox_inches="tight", pad_inches=0.1)
        plt.close(fig)
        print(f"Saved: {out_path}")


In [5]:
image_grounding_paths = checkpoint["image_grounding_paths"]
text_groundings = checkpoint["text_grounding"]


save_grounding_rows(image_grounding_paths, text_groundings, save_dir="/mnt/abka03/Projects/xl-vlms/outputs/imagenet_5_class_qwen_dl_imagenet_unsupervised_description/plots/concept_image",max_rows=1000)

Saved: /mnt/abka03/Projects/xl-vlms/outputs/imagenet_5_class_qwen_dl_imagenet_unsupervised_description/plots/concept_image/beaver_6207.pdf
Saved: /mnt/abka03/Projects/xl-vlms/outputs/imagenet_5_class_qwen_dl_imagenet_unsupervised_description/plots/concept_image/dog_5794.pdf
Saved: /mnt/abka03/Projects/xl-vlms/outputs/imagenet_5_class_qwen_dl_imagenet_unsupervised_description/plots/concept_image/dog_4521.pdf
Saved: /mnt/abka03/Projects/xl-vlms/outputs/imagenet_5_class_qwen_dl_imagenet_unsupervised_description/plots/concept_image/horse_4732.pdf
Saved: /mnt/abka03/Projects/xl-vlms/outputs/imagenet_5_class_qwen_dl_imagenet_unsupervised_description/plots/concept_image/bear_3302.pdf
Saved: /mnt/abka03/Projects/xl-vlms/outputs/imagenet_5_class_qwen_dl_imagenet_unsupervised_description/plots/concept_image/cat_3839.pdf
Saved: /mnt/abka03/Projects/xl-vlms/outputs/imagenet_5_class_qwen_dl_imagenet_unsupervised_description/plots/concept_image/tiger_5807.pdf
Saved: /mnt/abka03/Projects/xl-vlms/outp

In [2]:
import json
import ast
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.patches as patches
import os


def visualize_result_per_token(result, idx, save_dir="outputs", max_crops=5):
    """
    Visualize per-token concepts with bounding boxes and text_grounding labels.
    """
    os.makedirs(save_dir, exist_ok=True)

    # Load input image
    img_path = result["image_path"]
    img = mpimg.imread(img_path)

    ground_truth = result.get("ground_truth", "")
    prediction = result.get("model_output", "")

    # Number of tokens
    n_tokens = len(result.get("per_token_concepts", []))

    # Layout: 1 col for input + 3 concepts per token (each bar + max_crops images)
    ncols = 1 + 3 * (max_crops + 1)
    fig = plt.figure(figsize=(6 + (max_crops+1)*2, n_tokens*3.5))
    gs = fig.add_gridspec(
        n_tokens, ncols,
        width_ratios=[2] + [1]*(ncols-1),
        wspace=0.05, hspace=0.6
    )

    # Left side: input image
    ax_left = fig.add_subplot(gs[:, 0])
    ax_left.imshow(img)
    ax_left.axis("off")

    # Right side: per-token concepts
    for row, token in enumerate(result.get("per_token_concepts", [])):
        token_text = token.get("token_text", f"Token {row}")

        # token label on left side of row
        ax_row = fig.add_subplot(gs[row, 1:])  # dummy invisible axis spanning row
        ax_row.axis("off")
        ax_row.text(-0.05, 0.5, f"Token: {token_text}",
                    ha="right", va="center", fontsize=14, color="darkred",
                    transform=ax_row.transAxes)

        for concept_rank, concept in enumerate(token.get("top_concepts", [])[:3]):
            paths_str = concept["image_grounding_path"]
            paths = ast.literal_eval(paths_str) if isinstance(paths_str, str) else paths_str
            col_offset = 1 + concept_rank*(max_crops+1)

            # Similarity bar
            ax_bar = fig.add_subplot(gs[row, col_offset])
            sim = concept.get("similarity", 0.0)
            ax_bar.bar([0], [sim], color="orange", width=0.2)
            ax_bar.set_ylim(0, 1)
            ax_bar.set_xticks([])
            ax_bar.set_yticks([])
            ax_bar.set_title(f"R{concept_rank+1}", fontsize=8)

            # Collect axes for bounding box
            concept_axes = [ax_bar]

            # Concept crops
            for j, item in enumerate(paths[:max_crops]):
                try:
                    _, crop_path = item.split("@")
                except ValueError:
                    crop_path = item

                ax = fig.add_subplot(gs[row, col_offset + j + 1])
                if os.path.exists(crop_path):
                    crop_img = mpimg.imread(crop_path)
                    ax.imshow(crop_img)
                ax.axis("off")
                concept_axes.append(ax)

            # Draw bounding box + label
            fig.canvas.draw()
            bbox = [ax.get_position() for ax in concept_axes]
            x0 = min([b.x0 for b in bbox])
            y0 = min([b.y0 for b in bbox])
            x1 = max([b.x1 for b in bbox])
            y1 = max([b.y1 for b in bbox])

            
            # Add bounding box
            rect = patches.Rectangle(
                (x0, y0), x1-x0, y1-y0,
                transform=fig.transFigure, clip_on=False,
                linewidth=2, edgecolor="blue", facecolor="none", alpha=0.6
            )
            fig.patches.append(rect)

            # Add text_grounding label just below the bounding box
            text_label = ", ".join(concept.get("text_grounding", []))
            y_offset = (y1 - y0) * 0.05   # 5% of box height
            fig.text((x0+x1)/2, y0 - y_offset, text_label,
                    ha="center", va="top", fontsize=9, color="blue", wrap=True)


    plt.subplots_adjust(wspace=0.05, hspace=0.5)
    save_path = os.path.join(save_dir, f"per_token_viz_{idx}.png")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved per-token visualization: {save_path}")


def visualize_all_per_token(json_path, save_dir="outputs", max_crops=5):
    """
    Loop through all results in the JSON and visualize them (per token).
    """
    with open(json_path, "r") as f:
        data = json.load(f)

    if isinstance(data, dict) and "results" in data:
        results = data["results"]
    elif isinstance(data, list):
        results = data
    else:
        raise ValueError("JSON must be a list or have a 'results' key")

    for idx, result in enumerate(results):
        visualize_result_per_token(result, idx, save_dir, max_crops=max_crops)


In [4]:
visualize_all_per_token(
    "/mnt/abka03/Projects/xl-vlms/outputs/10_fruits_cdgl/explanations/snmf/vlm_explanations.json",
    save_dir="/mnt/abka03/Projects/xl-vlms/outputs/10_fruits_cdgl/plots/grounding_per_token",
    max_crops=5   # change this to 3, 10, etc.
)


Saved per-token visualization: /mnt/abka03/Projects/xl-vlms/outputs/10_fruits_cdgl/plots/grounding_per_token/per_token_viz_0.png
Saved per-token visualization: /mnt/abka03/Projects/xl-vlms/outputs/10_fruits_cdgl/plots/grounding_per_token/per_token_viz_1.png
Saved per-token visualization: /mnt/abka03/Projects/xl-vlms/outputs/10_fruits_cdgl/plots/grounding_per_token/per_token_viz_2.png
Saved per-token visualization: /mnt/abka03/Projects/xl-vlms/outputs/10_fruits_cdgl/plots/grounding_per_token/per_token_viz_3.png
Saved per-token visualization: /mnt/abka03/Projects/xl-vlms/outputs/10_fruits_cdgl/plots/grounding_per_token/per_token_viz_4.png
Saved per-token visualization: /mnt/abka03/Projects/xl-vlms/outputs/10_fruits_cdgl/plots/grounding_per_token/per_token_viz_5.png
Saved per-token visualization: /mnt/abka03/Projects/xl-vlms/outputs/10_fruits_cdgl/plots/grounding_per_token/per_token_viz_6.png
Saved per-token visualization: /mnt/abka03/Projects/xl-vlms/outputs/10_fruits_cdgl/plots/groundin